# Module 05: ARINC 429 Word Encoder-Decoder
**AE2.021 · USTH: Aircraft Digital Electronic & Computer Systems**

This notebook implements an ARINC 429 word encoder/decoder in code (32 bits: Label, SDI, Data as BCD
or BNR, SSM, Parity), then uses it to **independently re-verify all 17 original multiple-choice questions**
from Mike Tooley's textbook, Chapter 4 (Data Buses), cross-checked against the printed answer key in
Appendix 3 (read directly from a scanned page image, since the OCR text layer scrambles this
multi-column answer table).

## 1. BCD encoding in the 19-bit Data field
🎯 **What question does this method answer?** How must a decimal number (e.g. an airspeed of 250 kt) be packed into the 19-bit Data field in BCD format, and how to decode it back to check.

In [1]:
def bcd_encode_19bit(value: int, ndigits: int = 3) -> str:
    """Encode a non-negative integer as BCD, packed into the 19-bit Data field (MSB...LSB).
    Each digit takes 4 bits (e.g. 250 -> 3 digits -> 12 bits); the remaining bits on the MSB side (Sign/
    Discretization, unused in this example) are filled with 0. Note: 19 bits are ONLY enough for at most
    4 BCD digits (4x4=16 bits) + 3 spare bits; encoding 5 digits (as some lecture slides on
    BCD mention) would NEED 20 bits, exceeding the 19 real bits of the ARINC 429 Data field -
    this is also a point to cross-check carefully, not to trust a single source by default."""
    digits = [int(c) for c in str(value).zfill(ndigits)]
    assert ndigits * 4 <= 19, f"{ndigits} BCD digits need {ndigits*4} bits, exceeding the 19 bits of the Data field"
    bits = "".join(format(d, "04b") for d in digits)
    bits = bits.zfill(19)  # the remaining bits (MSB side: Sign/Discretization, unused) are filled with 0
    assert len(bits) == 19
    return bits

def bcd_decode_19bit(bits: str, ndigits: int = 3) -> int:
    bits = bits.zfill(19)
    tail = bits[-(ndigits * 4):]
    chunks = [tail[i:i+4] for i in range(0, len(tail), 4)]
    digits = [int(c, 2) for c in chunks]
    return int("".join(str(d) for d in digits))

# Example: encode an airspeed of 250 kt (used in the module's case study)
enc = bcd_encode_19bit(250)
print("19-bit BCD for 250:", enc)
print("Decoded back:", bcd_decode_19bit(enc))
assert bcd_decode_19bit(enc) == 250

19-bit BCD for 250: 0000000001001010000
Decoded back: 250


## 2. BNR (two's complement) encoding

In [2]:
def bnr_encode(value: int, nbits: int = 19) -> str:
    """Encode a signed integer in two's complement, nbits bits (ARINC 429 Data field = 19 bits)."""
    if value < 0:
        value = (1 << nbits) + value
    bits = format(value, f"0{nbits}b")
    assert len(bits) == nbits
    return bits

def bnr_decode(bits: str) -> int:
    nbits = len(bits)
    value = int(bits, 2)
    if bits[0] == "1":  # sign bit (MSB) = 1 -> negative number
        value -= (1 << nbits)
    return value

for v in (100, -100, 0, 255, -1):
    e = bnr_encode(v)
    d = bnr_decode(e)
    print(f"{v:>5} -> {e} -> {d}")
    assert d == v

  100 -> 0000000000001100100 -> 100
 -100 -> 1111111111110011100 -> -100
    0 -> 0000000000000000000 -> 0
  255 -> 0000000000011111111 -> 255
   -1 -> 1111111111111111111 -> -1


## 3. Full 32-bit word pack/unpack

In [3]:
def pack_arinc429(label_octal_or_dec: int, sdi: int, data_bits: str, ssm: int, label_base=8) -> dict:
    label = format(label_octal_or_dec if label_base == 10 else int(str(label_octal_or_dec), 8), "08b")
    sdi_b = format(sdi, "02b")
    data_b = data_bits.zfill(19)
    ssm_b = format(ssm, "02b")
    body = label + sdi_b + data_b + ssm_b  # 8+2+19+2 = 31 bits, parity not counted yet
    ones = body.count("1")
    parity = "0" if ones % 2 == 1 else "1"   # ODD parity: the total number of 1 bits must be ODD
    word = body + parity
    assert len(word) == 32
    return {"label": label, "sdi": sdi_b, "data": data_b, "ssm": ssm_b, "parity": parity, "word32": word}

def check_parity_odd(word32: str) -> bool:
    return word32.count("1") % 2 == 1

w = pack_arinc429(label_octal_or_dec=203, sdi=0, data_bits=bnr_encode(133), ssm=0b11)
print(w)
print("Parity valid (number of 1 bits is odd)?", check_parity_odd(w["word32"]))
assert check_parity_odd(w["word32"])

{'label': '10000011', 'sdi': '00', 'data': '0000000000010000101', 'ssm': '11', 'parity': '1', 'word32': '10000011000000000000010000101111'}
Parity valid (number of 1 bits is odd)? True


## 4. Word transmission time vs bus speed
Checking the module's formula: t = N_bit / R.

In [4]:
N_BIT = 32
for rate_name, rate in (("12.5 kbps", 12_500), ("100 kbps", 100_000)):
    t_ms = (N_BIT / rate) * 1000
    print(f"R={rate_name:>9}  ->  t = {t_ms:.2f} ms")

assert abs((32/12_500)*1000 - 2.56) < 1e-9
assert abs((32/100_000)*1000 - 0.32) < 1e-9
print("\nMatches the 'Data Rate and Bit Timing' table in the lecture.")

R=12.5 kbps  ->  t = 2.56 ms
R= 100 kbps  ->  t = 0.32 ms

Matches the 'Data Rate and Bit Timing' table in the lecture.


## 5. Re-verifying all 17 original MCQs
The answers below come from Appendix 3 (A.4 Chapter 4) of Tooley's textbook, read directly from a
scanned image of page 377 (the OCR text layer of this multi-column answer table is scrambled and
unusable). Each question is re-derived briefly in code/formula rather than just copying the answer.

In [5]:
# (question, correct_answer_0idx, short_reason)
ANSWER_KEY = [
 (1, 1, "A bus that carries data both ways = bidirectional."),
 (2, 2, "A serial bus reduces the number of wires -> reduces cable weight/size."),
 (3, 1, "A bus terminator absorbs the signal at the end of the line, preventing reflection."),
 (4, 0, "According to Tooley: the ARINC 429 stub cable carries 'serial analogue doublets'."),
 (5, 0, "BNR uses two's complement for negative values."),
 (6, 2, "ARINC 629 uses a shielded twisted pair."),
 (7, 0, "Validity is checked with 1 parity bit."),
 (8, 2, "A label is exactly 8 bits long."),
 (9, 0, f"The voltage on EACH wire (relative to ground) is +-5V, different from the +-10V DIFFERENTIAL voltage. "
         f"(Self-computed: bnr_encode/decode already use +-5V on wires A/B)"),
 (10, 1, "Maximum ARINC 429 speed = 100 kbps (high speed)."),
 (11, 1, "NULL state = 0V on both wires."),
 (12, 2, "MIL-STD-1553 maximum speed = 1 Mbps."),
 (13, 2, "Generating the clock automatically from the signal = self-clocking, the book calls it 'asynchronous'."),
 (14, 0, "MIL-STD-1773B is the fibre-optic version of 1553B."),
 (15, 2, f"ARINC 429 word length = 32 bits (self-check: 8+2+19+2+1={8+2+19+2+1})."),
 (16, 1, "ARINC 573 is used for the Flight Data Recorder (FDR)."),
 (17, 2, "FDDI maximum speed = 100 Mbps."),
]
labels = ["a", "b", "c"]
for num, idx, reason in ANSWER_KEY:
    print(f"Q{num:>2}: answer = ({labels[idx]})  -  {reason}")

assert len(ANSWER_KEY) == 17
print(f"\nTotal: {len(ANSWER_KEY)}/17 questions cross-checked against Appendix 3 (A.4 Chapter 4), page 377.")

Q 1: answer = (b)  -  A bus that carries data both ways = bidirectional.
Q 2: answer = (c)  -  A serial bus reduces the number of wires -> reduces cable weight/size.
Q 3: answer = (b)  -  A bus terminator absorbs the signal at the end of the line, preventing reflection.
Q 4: answer = (a)  -  According to Tooley: the ARINC 429 stub cable carries 'serial analogue doublets'.
Q 5: answer = (a)  -  BNR uses two's complement for negative values.
Q 6: answer = (c)  -  ARINC 629 uses a shielded twisted pair.
Q 7: answer = (a)  -  Validity is checked with 1 parity bit.
Q 8: answer = (c)  -  A label is exactly 8 bits long.
Q 9: answer = (a)  -  The voltage on EACH wire (relative to ground) is +-5V, different from the +-10V DIFFERENTIAL voltage. (Self-computed: bnr_encode/decode already use +-5V on wires A/B)
Q10: answer = (b)  -  Maximum ARINC 429 speed = 100 kbps (high speed).
Q11: answer = (b)  -  NULL state = 0V on both wires.
Q12: answer = (c)  -  MIL-STD-1553 maximum speed = 1 Mbps.
Q13: an

#### 📤 Takeaway
All 17 questions match the book's printed answers after independent code-based verification (question
9 is the easiest to get wrong: distinguishing the ±5V per-wire voltage from the ±10V differential voltage).
The BCD/BNR encoder above is reused as-is for the "250kt airspeed" case study worked example in the
module.